In [1]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import os
from collections import defaultdict
import math

moi10_adaptive_muts = ['1664AG', '1688GT', '1735CT', '1744AG', '1829TC','535AG'] <br>
moi_01_adaptive_muts = ['1697AC', '1691TC', '1685CT', '1649TC']<br>
founders_moi_01 = ['224CT','3299CT','1554GA','522GA']   founder mutations from Itamar's paper (MOI-0.1)

### process mutation types df pipeline:

In [2]:
def read_and_prepare_mut_types(mut_types_path, cols_to_keep=['ref_pos', 'read_base', 'ref_base','protein', 'mutation_type']):
    mutation_types = pd.read_csv(mut_types_path)
    mutation_types = mutation_types.rename(columns={'Pos':'ref_pos', 'Base':'read_base', 'Ref': 'ref_base'})
    mutation_types = mutation_types.loc[:,mutation_types.columns.isin(cols_to_keep)]
    mutation_types['mut_name'] = mutation_types['ref_pos'].astype(str)+mutation_types['ref_base']+mutation_types['read_base']
    mutation_types['syn_nonsyn'] = np.where(mutation_types['mutation_type'].isna(), 'syn', 'nonsyn') # add syn_nonsyn column
    mutation_types['stop_codon'] = mutation_types['mutation_type'].str.contains('\\*', na=False) # add stop_codon column
    return mutation_types

def filter_out_overlapping_syn_mutations(mut_type_df):
    """
    this function adds a column named 'keep' with boolian values.
    if True -- the mutation is either nonoverlapping mutation or a nonsyn overlapping mutation.
    if False -- this mutation affects 2 genes, and this instance is syn.
    (the mutations [1791TC, 1902TC, 1904AG] overlap and are syn in both proteins, they all get False in 'keep' column)
    
    the function also adds a 'is_duplicate' column -- points the mutations that affect 2 genes. 
    returns a new mut_type_df without the duplicated syn mutations, with another column that shows if this mutation affects
    2 genes. 
    """
    df_copy = mut_type_df.copy()
    df_copy['priority'] = df_copy['syn_nonsyn'].map({'nonsyn': 1, 'syn': 0})
    df_sorted = df_copy.sort_values(['mut_name', 'priority'], ascending=[True, False]) 
    df_sorted['is_duplicate'] = df_sorted.duplicated(subset=['mut_name'],keep=False)
    df_sorted['keep'] = np.where((df_sorted['priority']==0)&(df_sorted['is_duplicate']==True),False, True)
    result_mutype_df = df_sorted.drop(['priority'], axis=1)
    result_mutype_df = result_mutype_df.reset_index(drop=True)
    result_mutype_df = result_mutype_df[result_mutype_df['keep']==True]
    result_mutype_df = result_mutype_df.drop(['keep','read_base', 'ref_base'],axis=1)
    return result_mutype_df

def add_adaptive_col_byMOI_to_mutypes(mut_types_df):
    # adaptive mutations according to serrial passaging experiments 
    moi10_adaptive_muts = ['1664AG', '1688GT', '1735CT', '1744AG', '1829TC','535AG']
    moi_01_adaptive_muts = ['1697AC', '1691TC', '1685CT', '1649TC']
    
    moi10_weak_def_adaptive_muts = ['1549CT','1560GA','1664AG','1688GC','1688GT','1728TC','1735CT','1744AG','1819CT','1829TC','2572TG','2859CT','535AG']
    moi01_weak_def_adaptive_muts =['1554GA', '1571TG', '1649TC', '1652TC', '1655GA', '1685CT', '1691TC', '1697AC', '224CT', '3299CT', '522GA']

    mut_types_df['adaptive_moi10'] = np.where(mut_types_df['mut_name'].isin(moi10_adaptive_muts),'ada','nonada')
    mut_types_df['adaptive_moi0.1'] = np.where(mut_types_df['mut_name'].isin(moi_01_adaptive_muts),'ada','nonada')
    
    mut_types_df['weak_adaptive_moi10'] = np.where(mut_types_df['mut_name'].isin(moi10_weak_def_adaptive_muts),'ada','nonada')
    mut_types_df['weak_adaptive_moi0.1'] = np.where(mut_types_df['mut_name'].isin(moi01_weak_def_adaptive_muts),'ada','nonada')
    return mut_types_df

In [3]:
def process_mutation_types_df(mut_types_path):
    mut_types_df = filter_out_overlapping_syn_mutations(read_and_prepare_mut_types(mut_types_path))
    return add_adaptive_col_byMOI_to_mutypes(mut_types_df)    

In [4]:
mut_types_path = '/sternadi/home/volume2/noam/ms2/ms2_reference/mutation_type_ncbi/mutation_types_short.csv'
mut_types_df = process_mutation_types_df(mut_types_path)

In [5]:
def add_cols_and_concat(df, df_path):
    if 'A-carmel' in df_path:
        line = 'A'
        moi = 10
        passage = 10
    elif 'B-maria' in df_path:
        line = 'G'
        moi = 10
        passage = 10
    elif 'C-maria' in df_path:
        line = 'H'
        moi = 10
        passage = 10
    elif 'C-shir' in df_path:
        line = 'E'
        moi = 10
        passage = 10
    else:
        moi=0.1
        df['MOI'] =  df.get('MOI', moi)
        return df
    
    df['line'] = df.get('line', line)
    df['passage'] =  df.get('passage', passage)
    df['MOI'] =  df.get('MOI', moi)

    return df 


def read_files(reads_lst_paths, mutations_lst_paths, freqs_lst_path=[], read_freqs=False, 
               cols_to_keep =['read_id','mut_name','mut_num','position','mutations','mutation','type','passage','line','MOI']):
    reads_df_lst = []
    for reads_path in reads_lst_paths:
        sep = '\t' if 'tsv' in reads_path else ','
        reads_df = pd.read_csv(reads_path, sep=sep)
        reads_df_lst.append(add_cols_and_concat(reads_df, reads_path))
    all_reads_df = pd.concat(reads_df_lst)
    mutations_df_lst = []
    for muts_path in mutations_lst_paths:
        sep = '\t' if 'tsv' in muts_path else ','
        muts_df = pd.read_csv(muts_path, sep=sep)
        mutations_df_lst.append(add_cols_and_concat(muts_df, muts_path))
    all_mutations_df = pd.concat(mutations_df_lst)
    if read_freqs:
        freqs_df_lst = []
        for freqs_path in freqs_lst_path:
            sep = '\t' if 'tsv' in freqs_path else ','
            freqs_df = pd.read_csv(freqs_path, sep=sep)
            freqs_df_lst.append(add_cols_and_concat(freqs_df, freqs_path))
        all_freqs_df = pd.concat(freqs_df_lst)
    
    all_reads_df = all_reads_df.loc[:,all_reads_df.columns.isin(cols_to_keep)]
    all_mutations_df = all_mutations_df.loc[:,all_mutations_df.columns.isin(cols_to_keep)]
    if read_freqs:
        all_freqs_df = all_freqs_df.loc[:,all_freqs_df.columns.isin(cols_to_keep)]
        return all_reads_df, all_mutations_df, all_freqs_df
        
    return all_reads_df, all_mutations_df
        
    
def sample_from_data(reads_df, mutations_df, sample_size=1309):
    '''
    coverage (number of reads in each line) is different between lines and MOIs, sampling same number of data points from 
    each line should help overcome this bias.
    samples n (=sample size) reads from reads_df and takes the coresponding mutataions from mutations_df
    retruns tuple df sampled reads and sampled mutations.
    '''
    if not {'passage', 'line', 'read_id'}.issubset(reads_df.columns):
        raise ValueError("reads_df must have 'passage' and 'line' columns for sampling.")
    reads_df_sampled = reads_df.groupby(['passage', 'line'], group_keys=False).apply(
    lambda x: x.sample(n=sample_size, random_state=42, replace=False)).reset_index(drop=True)
    
    if not {'read_id'}.issubset(mutations_df.columns):
        raise ValueError("mutations_df must have 'read_id' for matching the reads samples.")
    mutations_df_sampled = mutations_df[(mutations_df['read_id'].isin(reads_df_sampled['read_id']))].reset_index(drop=True)
    
    return reads_df_sampled, mutations_df_sampled
    

def filter_out_indels_and_founders(mutations_df, is_moi01):
    founders = ['224CT', '3299CT', '1554GA', '522GA'] if is_moi01 else []
    mutations_df_filt = mutations_df[(~mutations_df['mut_name'].str.contains('-'))&(~mutations_df['mut_name'].isin(founders))].reset_index(drop=True)
    return mutations_df_filt
    
    
    
def create_freq_files(mutations_df_sampled, sample_size=1309):
    freq_df_sampled = (mutations_df_sampled.groupby(['mut_name', 'position', 'mutation', 'line', 'passage'])
        .agg(read_count=('read_id', 'nunique'))
        .reset_index()
    )

    # Calculate frequency per line and passage
    freq_df_sampled['freq'] = freq_df_sampled.groupby(['line', 'passage'])['read_count'].transform(lambda x: x / sample_size)
    return freq_df_sampled
    

    

In [6]:
def add_metadata_to_muts(mutations_df, mut_types, moi):
    '''
    add to mutations_df the columns: protein, syn_nonsyn, adaptive_moiXX
    Arguments:
        mutations_df - pandas df, after sampling and filtering out indels (and founders if needed - moi0.1). 
        can also be freqs_df!
        mut_types - pandas df, the output df of the process mutation type pipeline,
        with the columns: [ref_pos, protein, mut_name, syn_nonsyn, is_duplicate, adaptive_moi10, adaptive_moi01]
        moi - str, 10 or 0.1, specifies the adaptive column that should be used in mut_types.
    
    '''
    if (moi!='10') and (moi!='0.1'):
        raise ValueError("moi gets only the values (str) 10 or 0.1")
       
    adaptive_col = mut_types.columns[mut_types.columns.str.contains(moi)][0]
    mini_mut_types = mut_types[['protein', 'mut_name', 'syn_nonsyn','stop_codon', adaptive_col, 'weak_'+adaptive_col]]
    mutations_with_metadata = pd.merge(mutations_df, mini_mut_types, on='mut_name')
    return mutations_with_metadata

In [7]:
def process_reads_and_mutations(moi, reads_lst_paths, mutations_lst_paths, freqs_lst_path, sample, sample_size, mut_types_df):
    '''
    load reads file and mutations file (long reads)
    for MOI10 data - add line and passage and concat lines 
    sample n reads (to overcome coverage bias) and use according mutations 
    filter out indels (and founders for MOI0.1 only)
    create new freq files if sampling is on, otherwise load the original freq file
    use mut_types_df to add metadata to mutations df
    
    Arguments:
    moi - str, specifies the MOI, can get the values '10' or '0.1'
    reads/mutations/freqs_lst_paths - list, path to files
    sample - bool, if True sample sample_size from reads, and read_freqs will be defined as False 
    (do not use original freq files), if False - do not sample from reads and use original freq files 
    sample_size - int, size to sample 
    '''
    if (moi!='10') and (moi!='0.1'):
        raise ValueError("moi gets only the values (str) 10 or 0.1")
    read_freqs = 1-sample
    is_moi01 = True if moi=='0.1' else False
    if read_freqs:
        reads_df, muts_df, freqs_df = read_files(reads_lst_paths,mutations_lst_paths, freqs_lst_path, read_freqs)
    else:
        reads_df, muts_df = read_files(reads_lst_paths,mutations_lst_paths)
    if sample:
        reads_df, muts_df = sample_from_data(reads_df, muts_df, sample_size)
    muts_filt = filter_out_indels_and_founders(muts_df, is_moi01)
    muts_with_metadata = add_metadata_to_muts(muts_filt, mut_types_df, moi)
    if not read_freqs:
        freqs_df = create_freq_files(muts_with_metadata, sample_size)
    freqs_with_metadata = add_metadata_to_muts(freqs_df, mut_types_df, moi)
    reads_df.name = f'MOI{moi}_reads_df'
    muts_with_metadata.name = f'MOI{moi}_muts_with_metadata'
    freqs_with_metadata.name = f'MOI{moi}_freqs_with_metadata'
    return reads_df, muts_with_metadata, freqs_with_metadata

In [8]:
def save_df_as_csv(df, file_name, path):
    if not os.path.exists(path):
        raise ValueError("path does not exist!")
    df.to_csv(f'{path}/{file_name}.csv', index=False)
    print(f'saved data frame: {df.name} to path: {path} as the file: {file_name}.csv')

In [9]:
# MOI-10
com_path = '/sternadi/nobackup/volume1/arielle/loop_results/moi_10/'
a_carmel = 'L4f6a475f_sample_p10-A-carmel-loop_contig_list_trimmed/'
b_maria = 'L4f6a475f_sample_p10-B-maria-loop_contig_list_trimmed/'
c_maria = 'L4f6a475f_sample_p10-C-maria-loop_contig_list_trimmed/'
c_shir = 'L4f6a475f_sample_p10-C-shir-loop_contig_list_trimmed/'
reads = 'filtered_reads.tsv'
mutations = 'mutations.tsv'
freqs = 'freqs.tsv'
reads_lst_paths = [com_path+a_carmel+reads, com_path+b_maria+reads,com_path+c_maria+reads,com_path+c_shir+reads]
freqs_lst_paths = [com_path+a_carmel+freqs, com_path+b_maria+freqs, com_path+c_maria+freqs, com_path+c_shir+freqs]
muts_lst_paths = [com_path+a_carmel+mutations, com_path+b_maria+mutations, com_path+c_maria+mutations, com_path+c_shir+mutations]

In [10]:
reads_sampled_df, muts_with_metadata_df, freqs_with_metadata_df = process_reads_and_mutations('10', reads_lst_paths, muts_lst_paths, freqs_lst_paths, True, 1309, mut_types_df)

In [15]:
# save_df_as_csv(reads_sampled_df,'moi10_reads_sampled_and_filtered', '/sternadi/home/volume3/yael/Data/MOI10')  # NOTICE: reads file is not filtered at this point, only happens in enrich_reads
# save_df_as_csv(muts_with_metadata_df,'moi10_muts_w_metadata_sampled_and_filtered_with2ada_def', '/sternadi/home/volume3/yael/Data/MOI10')
# save_df_as_csv(freqs_with_metadata_df,'moi10_new_freqs_w_metadata_filtered_with2ada_def', '/sternadi/home/volume3/yael/Data/MOI10')

saved data frame: MOI10_reads_df to path: /sternadi/home/volume3/yael/Data/MOI10 as the file: moi10_reads_sampled_and_filtered.csv
saved data frame: MOI10_muts_with_metadata to path: /sternadi/home/volume3/yael/Data/MOI10 as the file: moi10_muts_w_metadata_sampled_and_filtered_with2ada_def.csv
saved data frame: MOI10_freqs_with_metadata to path: /sternadi/home/volume3/yael/Data/MOI10 as the file: moi10_new_freqs_w_metadata_filtered_with2ada_def.csv


In [11]:
# MOI-0.1
com_path_moi01 = '/sternadi/home/volume2/ita/ms2-mutation-rate/analysis/MOI01_long_reads_'
reads_01 = 'reads.tsv'
muts_01 = 'mutations.tsv'
freqs_01 = 'freqs.tsv'

reads_lst_path_01 = [com_path_moi01+reads_01]
freqs_lst_path_01 = [com_path_moi01+freqs_01]
muts_lst_path_01 = [com_path_moi01+muts_01]

In [12]:
reads_sampled_df_01, muts_with_metadata_df_01, freqs_with_metadata_df_01 = process_reads_and_mutations('0.1', reads_lst_path_01, muts_lst_path_01, freqs_lst_path_01, True, 1309, mut_types_df)

In [13]:
# save_df_as_csv(reads_sampled_df_01,'moi01_reads_sampled', '/sternadi/home/volume3/yael/Data/MOI01')
# save_df_as_csv(muts_with_metadata_df_01,'moi01_muts_w_metadata_sampled_and_filtered_with2ada_def', '/sternadi/home/volume3/yael/Data/MOI01')
# save_df_as_csv(freqs_with_metadata_df_01,'moi01_new_freqs_w_metadata_filtered_with2ada_def', '/sternadi/home/volume3/yael/Data/MOI01')

In [14]:
def enrich_reads_with_mutations(
    reads_df: pd.DataFrame,
    mutations_with_metadata_df: pd.DataFrame,
    adaptive_def: str = 'strong',
    min_read_count: int = 0,
    proteins = ('mat', 'cp', 'lys', 'rep'),
    keep_cols = ('read_id', 'mut_num', 'mutations', 'line','passage','MOI'),
) -> pd.DataFrame:
    """
    Build a per-read enriched table by joining reads with their mutations and
    deriving per-protein syn/nonsyn/ada counts, filtered by a read-count threshold.

    Parameters
    ----------
    reads_df : DataFrame
        Columns: ['read_id', 'mut_num', 'mutations', 'line'].
    mutations_with_metadata_df : DataFrame
        Columns: ['mutation','position','read_id','type','mut_name','line',
                  'passage','MOI','protein' (mat/cp/lys/rep),
                  'syn_nonsyn' (syn/nonsyn),
                  'adaptive_moiXX' (ada/nonada)] (where XX is 10 or 0.1 depends whether the df is of moi10 or 0.1)
    adaptive_def: str, default 'strong'. 'strong' refers to 'adaptive_moi_X', where a mutation is considered adaptive if it passes 3%
        in all lines. 'weak' refers to 'weak_adaptive_moi_X', where a mutations is considered adaptive if it passes 
        1% in at least 2 lines. 
    min_read_count : int, default 0
        Filter out mutations whose per-(mut_name, line) count is < min_read_count.
        (Mutations not present in mutations_df are also filtered out.)
    proteins : iterable of str
        Protein names to create facet/count columns for.
    keep_cols : iterable of str
        Columns to carry through from reads_df.

    Returns
    -------
    DataFrame
        Columns:
        ['read_id','mut_num','mutations','line','mut_name','protein','syn_nonsyn','adaptive',
         'mat_syn_nonada','mat_syn_ada','mat_nonsyn_nonada','mat_nonsyn_ada',
         'cp_syn_nonada','cp_syn_ada','cp_nonsyn_nonada','cp_nonsyn_ada',
         'lys_syn_nonada','lys_syn_ada','lys_nonsyn_nonada','lys_nonsyn_ada',
         'rep_syn_nonada','rep_syn_ada','rep_nonsyn_nonada','rep_nonsyn_ada',
         'mutations_filtered','mut_num_filtered','total_syn_nonada','total_syn_ada']
    """

    if not {'mut_name', 'line', 'passage'}.issubset(mutations_with_metadata_df.columns):
        raise ValueError("mutations_df must have 'mut_name' and 'line' columns for filtering.")

    counts = (
        mutations_with_metadata_df.groupby(['mut_name', 'line', 'passage'])
        .size()
        .rename('mut_read_count')
        .reset_index()
    )
    # filter out mutations that appear in less than min_reads threshold (if threshold 0 doesnt filter out mutations.)
    muts_filt = (
        mutations_with_metadata_df.merge(counts, on=['mut_name', 'line','passage'], how='left')
        .query('mut_read_count >= @min_read_count')
        .drop(columns='mut_read_count')
    )

    # aggregate filtered mutations per read_id 
    adap_col = muts_filt.columns[muts_filt.columns.str.contains('adaptive')][0]
    adaptive_col = adap_col if adaptive_def == 'strong' else 'weak_'+adap_col
    agg = (
        muts_filt.groupby("read_id", as_index=False)
        .agg({
            "mut_name": list,
            "protein": list,
            "syn_nonsyn": list,
            adaptive_col: list,
        })
    )

    # merge to reads and normalize list columns 
    # keep_cols = ('read_id', 'mut_num', 'mutations', 'line','passage','MOI')
    base_cols = [c for c in keep_cols if c in reads_df.columns]
    reads_enriched = reads_df[base_cols].merge(agg, on="read_id", how="left")

    for c in ["mut_name", "protein", "syn_nonsyn", adaptive_col]:
        if c not in reads_enriched.columns:
            reads_enriched[c] = [[] for _ in range(len(reads_enriched))]
        else:
            reads_enriched[c] = reads_enriched[c].apply(lambda x: x if isinstance(x, list) else [])

    # per-protein syn/nonsyn/ada counts
    for prot in proteins:
        for mut_type in ["syn", "nonsyn"]:
            for ada in ["nonada", "ada"]:
                col = f"{prot}_{mut_type}_{ada}"
                reads_enriched[col] = reads_enriched.apply(
                    lambda row: sum(
                        (p == prot) and (t == mut_type) and (a == ada)
                        for p, t, a in zip(row["protein"], row["syn_nonsyn"], row[adaptive_col])
                    ),
                    axis=1,
                )

    # keep only allowed mutations in 'mutations' string (order-sorted) 
    def _filter_muts(row):
        muts = row.get('mutations', '')
        if (muts is np.nan) or (muts == ''):
            return 'WT'
        tokens = muts.split('_')
        valid = set(row['mut_name'])  # only those that survived filtering of the aggregation of mutations (which are only mutations that are in mut_types - coding mutations and no indels)
        kept = sorted([tok for tok in tokens if tok in valid])
        return '_'.join(kept) if kept else 'WT'

    reads_enriched = reads_enriched.fillna('')
    reads_enriched['mutations_filtered'] = reads_enriched.apply(_filter_muts, axis=1)

    def _count_mut_num(s):
        return 0 if s in ('WT', '') else len(s.split('_'))

    reads_enriched['mut_num_filtered'] = reads_enriched['mutations_filtered'].apply(_count_mut_num)

    # totals across proteins 
    reads_enriched['total_syn_nonada'] = sum(reads_enriched[f"{p}_syn_nonada"] for p in proteins)
    reads_enriched['total_syn_ada'] = sum(reads_enriched[f"{p}_syn_ada"] for p in proteins)

    #  final column order 
    per_protein_cols = []
    for p in proteins:
        per_protein_cols += [
            f"{p}_syn_nonada", f"{p}_syn_ada",
            f"{p}_nonsyn_nonada", f"{p}_nonsyn_ada",
        ]

    final_cols = [
        'read_id','passage', 'line', 'mut_num', 'mutations',
        'mutations_filtered', 'mut_num_filtered',
        'mut_name', 'protein', 'syn_nonsyn', adaptive_col,
        *per_protein_cols,       
        'total_syn_nonada', 'total_syn_ada'
    ]

    # ensure all expected columns exist (in case some proteins/types absent)
    for c in final_cols:
        if c not in reads_enriched.columns:
            if c in ('mut_name', 'protein', 'syn_nonsyn', adaptive_col):
                reads_enriched[c] = [[] for _ in range(len(reads_enriched))]
            elif c.endswith(('_nonada', '_ada')) or c in ('mut_num_filtered','total_syn_nonada','total_syn_ada'):
                reads_enriched[c] = 0
            else:
                reads_enriched[c] = ''
    reads_enriched = reads_enriched[final_cols]
    reads_enriched.name = f'MOI{reads_df.MOI[0]}_reads_enriched_df'
    return reads_enriched

In [15]:
moi10_enriched_reads = enrich_reads_with_mutations(reads_sampled_df, muts_with_metadata_df)
moi01_enriched_reads = enrich_reads_with_mutations(reads_sampled_df_01, muts_with_metadata_df_01)

moi10_enriched_reads_weak_ada = enrich_reads_with_mutations(reads_sampled_df, muts_with_metadata_df,'weak')
moi01_enriched_reads_weak_ada = enrich_reads_with_mutations(reads_sampled_df_01, muts_with_metadata_df_01,'weak')

In [ ]:
# save_df_as_csv(moi10_enriched_reads,'moi10_enriched_reads_sampled_and_filtered', '/sternadi/home/volume3/yael/Data/MOI10')
save_df_as_csv(moi01_enriched_reads,'moi01_enriched_reads_sampled_and_filtered', '/sternadi/home/volume3/yael/Data/MOI01')

In [ ]:
# save_df_as_csv(moi10_enriched_reads_weak_ada,'moi10_enriched_reads_weak_ada_sampled_and_filtered', '/sternadi/home/volume3/yael/Data/MOI10')
save_df_as_csv(moi01_enriched_reads_weak_ada,'moi01_enriched_reads_weak_ada_sampled_and_filtered', '/sternadi/home/volume3/yael/Data/MOI01')

#### divide genomes into adaptive and nonadaptive

In [16]:
def add_adaptive_genome_column(reads_enriched):
    adaptive_col = reads_enriched.columns[reads_enriched.columns.str.contains('adaptive')][0]
    moi='10' if (adaptive_col[-2:]=='10') else '0.1'
    adaptive_genomes_mask = reads_enriched[adaptive_col].apply(lambda x: isinstance(x, list) and 'ada' in x)
    reads_enriched['adaptive_genome'] = adaptive_genomes_mask
    reads_enriched.name = f'MOI{moi}_reads_enriched'
    return reads_enriched


def create_freq_file(mutations_df_sampled, sample_size=1309):
    freq_df_sampled = (mutations_df_sampled.groupby(['mut_name', 'position', 'mutation', 'line', 'passage','adaptive_genome'])
        .agg(read_count=('read_id', 'nunique'))
        .reset_index()
    )

    # Calculate frequency per line and passage
    freq_df_sampled['freq'] = freq_df_sampled.groupby(['line', 'passage'])['read_count'].transform(lambda x: x / sample_size)
    return freq_df_sampled

In [17]:
moi10_enriched_reads = add_adaptive_genome_column(moi10_enriched_reads)
muts_with_metadata_10 = muts_with_metadata_df.merge(moi10_enriched_reads[['read_id','adaptive_genome']],on='read_id') # ,how='left')
muts_with_metadata_10.name = 'MOI10_muts_with_metadata'

freqs_ada_genome_sep_10 = create_freq_file(muts_with_metadata_10)
freqs_ada_genome_sep_10_w_md = add_metadata_to_muts(freqs_ada_genome_sep_10, mut_types_df, '10')
freqs_ada_genome_sep_10_w_md.name = 'MOI10_freqs_ada_genome_sep_w_md'

In [18]:
moi01_enriched_reads = add_adaptive_genome_column(moi01_enriched_reads)
muts_with_metadata_01 = muts_with_metadata_df_01.merge(moi01_enriched_reads[['read_id','adaptive_genome']],on='read_id') #,how='left')
muts_with_metadata_01.name = 'MOI01_muts_with_metadata'

freqs_ada_genome_sep_01 = create_freq_file(muts_with_metadata_01)
freqs_ada_genome_sep_01_w_md = add_metadata_to_muts(freqs_ada_genome_sep_01, mut_types_df, '0.1')
freqs_ada_genome_sep_01_w_md.name = 'MOI01_freqs_ada_genome_sep_w_md'

In [ ]:
# save_df_as_csv(reads_sampled_df,'moi10_reads_sampled_and_filtered', '/sternadi/home/volume3/yael/Data/MOI10')
save_df_as_csv(muts_with_metadata_10,'moi10_muts_processed', '/sternadi/home/volume3/yael/Data/MOI10')  # final version of muts
save_df_as_csv(freqs_ada_genome_sep_10_w_md,'moi10_freqs_ada_genome_sep', '/sternadi/home/volume3/yael/Data/MOI10')
save_df_as_csv(moi10_enriched_reads,'moi10_enriched_reads_processed', '/sternadi/home/volume3/yael/Data/MOI10')  # final version of enriched_reads_moi10

In [ ]:
# save_df_as_csv(reads_sampled_df_01,'moi01_reads_sampled_and_filtered', '/sternadi/home/volume3/yael/Data/MOI01')
save_df_as_csv(muts_with_metadata_01,'moi01_muts_processed', '/sternadi/home/volume3/yael/Data/MOI01')  # final version of muts
save_df_as_csv(freqs_ada_genome_sep_01_w_md,'moi01_freqs_ada_genome_sep', '/sternadi/home/volume3/yael/Data/MOI01')
save_df_as_csv(moi01_enriched_reads,'moi01_enriched_reads_processed', '/sternadi/home/volume3/yael/Data/MOI01')   # final version of enriched_reads_moi01